In [25]:
import csv
import nibabel as nib
import matplotlib.pyplot as plt
import random
import torch
import os
import numpy as np
from model1 import CNN_3D,NiiDataset,MultiModalTransformer,NeuralNet,TransEModel,KGMultiModalTransformer
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import nibabel as nib
import shutil
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import math
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd


In [26]:
path_existence = []
data_prodromal=[]
data_swedd=[]
data_control=[]
data_PD = []
count_control = 0
count_PD = 0
count_swedd = 0
count_prodromal = 0
with open('PD1.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  
    for row in csv_reader:
        path = 'PD/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_PD=count_PD+1
            data_PD.append(row)
with open('control.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'Control/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_control=count_control+1
            data_control.append(row)
with open('swedd.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'SWEDD/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_swedd=count_swedd+1
            data_swedd.append(row)
with open('prodromal.csv', mode='r', newline='', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader) 
    for row in csv_reader:
        path = 'Prodromal/' + row[1]
        exists = os.path.exists(path)
        path_existence.append((path, exists))
        if exists:
            count_prodromal=count_prodromal+1
            data_prodromal.append(row)
print(count_PD) 
print(count_control) 
print(count_swedd) 
print(count_prodromal)


125
132
72
80


In [27]:
PD_arrays=[]
replace_dict = {'F': '0','M':'1','Normal':'0','Slight':'1','Mild':'2','Moderate':'3','Severe':'4','T1-anatomical':'1','Processed':'1', 
               'BL':'0','V04':'1','V06':'2','V06':'3','V08':'4','V10':'5','NiFTI':'1','':'0','No':'0','Yes':'1','Stage 1':'1','Stage 2':'2','Stage 3':'3','Stage 4':'4'
               ,'On':'1','Off':'0','Stage 0':'0'}
for i in data_PD:
    j= i[21:]
    j= [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    PD_array = np.array(num_list)
    PD_arrays.append(PD_array)
PD_array = np.vstack(PD_arrays)

control_arrays = []
for i in data_control:
    j = i[21:]
    j = [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    control_array = np.array(num_list)
    control_arrays.append(control_array)
control_array = np.vstack(control_arrays)

swedd_arrays = []
for i in data_swedd:
    j = i[21:]
    j = [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    swedd_array = np.array(num_list)
    swedd_arrays.append(swedd_array)
swedd_array = np.vstack(swedd_arrays)

prodromal_arrays = []
for i in data_prodromal:
    j = i[21:]
    j = [replace_dict.get(item, item) for item in j]
    num_list = [float(num) for num in j]
    prodromal_array = np.array(num_list)
    prodromal_arrays.append(prodromal_array)
prodromal_array = np.vstack(prodromal_arrays)

#加权算值
def weighted_sum(tensor):
    weights = [0.1, 0.3, 0.5 , 0.7]
    weight_tensor = torch.tensor(weights, dtype=tensor.dtype, device=tensor.device)
    weighted_sum_result = torch.sum(tensor * weight_tensor, dim=1, keepdim=True)
    return weighted_sum_result



In [28]:
pd_tensor = torch.from_numpy(PD_array).float()
control_tensor = torch.from_numpy(control_array).float()
swedd_tensor = torch.from_numpy(swedd_array).float()
prodromal_tensor = torch.from_numpy(prodromal_array).float()

pd_labels = torch.zeros(pd_tensor.shape[0], dtype=torch.long)
control_labels = torch.ones(control_tensor.shape[0], dtype=torch.long)
swedd_labels = torch.full((swedd_tensor.shape[0],), 3, dtype=torch.long)
prodromal_labels = torch.full((prodromal_tensor.shape[0],), 2, dtype=torch.long)

X = torch.cat([pd_tensor, control_tensor, swedd_tensor,prodromal_tensor], dim=0)
y = torch.cat([pd_labels , control_labels  ,swedd_labels, prodromal_labels], dim=0)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = NeuralNet(embedding=64)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50
for epoch in range(num_epochs):
    for inputs, labels in dataloader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        
with torch.no_grad():
    pd_tensor = model(pd_tensor)
    control_tensor = model(control_tensor)
    swedd_tensor = model(swedd_tensor)
    prodromal_tensor = model(prodromal_tensor)
    
pd_tensor = weighted_sum(pd_tensor)
control_tensor = weighted_sum(control_tensor)
swedd_tensor = weighted_sum(swedd_tensor)
prodromal_tensor = weighted_sum(prodromal_tensor)

Epoch [10/50], Loss: 0.8228
Epoch [20/50], Loss: 0.4570
Epoch [30/50], Loss: 0.5588
Epoch [40/50], Loss: 0.5374
Epoch [50/50], Loss: 0.2841


In [29]:
# 数据处理函数
def preprocess_data(data, replace_dict):
    processed_data = []
    for row in data:
        row = [replace_dict.get(item, item) for item in row]
        row = [float(item) if item.replace('.', '', 1).isdigit() else item for item in row]
        processed_data.append(row[4:22])
    return np.array(processed_data)
# 编码类别型变量
def encode_categorical(data, categorical_indices):
    encoded_data = data.copy()
    for idx in categorical_indices:
        le = LabelEncoder()
        encoded_data[:, idx] = le.fit_transform(encoded_data[:, idx])
    return encoded_data.astype(float)

In [30]:
pd_data = preprocess_data(data_PD, replace_dict)
control_data = preprocess_data(data_control, replace_dict)
swedd_data = preprocess_data(data_swedd, replace_dict)
prodromal_data = preprocess_data(data_prodromal, replace_dict)

categorical_indices = [2, 3, 4, 5]  # gender, education, hispanic, race, apoe
pd_EHR = encode_categorical(pd_data, categorical_indices)
control_EHR = encode_categorical(control_data, categorical_indices)
swedd_EHR = encode_categorical(swedd_data, categorical_indices)
prodromal_EHR = encode_categorical(prodromal_data, categorical_indices)

pd_EHR = torch.from_numpy(pd_EHR).float()
control_EHR = torch.from_numpy(control_EHR).float()
swedd_EHR = torch.from_numpy(swedd_EHR).float()
prodromal_EHR = torch.from_numpy(prodromal_EHR).float()

linear_layer = nn.Linear(18, 16)
control_EHR = linear_layer(control_EHR)
pd_EHR = linear_layer(pd_EHR)
swedd_EHR = linear_layer(swedd_EHR)
prodromal_EHR = linear_layer(prodromal_EHR)

linear_layer = nn.Linear(16, 1)
control_EHR = linear_layer(control_EHR)
pd_EHR = linear_layer(pd_EHR)
swedd_EHR = linear_layer(swedd_EHR)
prodromal_EHR = linear_layer(prodromal_EHR)

print('pd.EHR--->', pd_EHR.shape)
print('control.EHR--->', control_EHR.shape)
print('swedd.EHR--->', swedd_EHR.shape)
print('prodromal.EHR--->', prodromal_EHR.shape)

pd.EHR---> torch.Size([125, 1])
control.EHR---> torch.Size([132, 1])
swedd.EHR---> torch.Size([72, 1])
prodromal.EHR---> torch.Size([80, 1])


In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
def center_crop_3d(tensor, size):
    depth, height, width = tensor.shape
    target_depth, target_height, target_width = size

    start_depth = (depth - target_depth) // 2
    start_height = (height - target_height) // 2
    start_width = (width - target_width) // 2

    end_depth = start_depth + target_depth
    end_height = start_height + target_height
    end_width = start_width + target_width

    return tensor[start_depth:end_depth, start_height:end_height, start_width:end_width]

In [33]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
import nibabel as nib

# 你的 NiiDataset，务必用这版
class NiiDataset(Dataset):
    def __init__(self, file_list_or_folder):
        if isinstance(file_list_or_folder, list):
            self.file_list = file_list_or_folder
        elif isinstance(file_list_or_folder, str):
            folder_path = file_list_or_folder
            self.file_list = [os.path.join(folder_path, filename)
                              for filename in os.listdir(folder_path)
                              if filename.endswith('.nii') or filename.endswith('.nii.gz')]
        else:
            raise ValueError("参数应为list或str")
    def __len__(self):
        return len(self.file_list)
    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        img = nib.load(file_path)
        img_data = img.get_fdata()
        img_tensor = torch.from_numpy(img_data).float()
        cropped_tensor = center_crop_3d(img_tensor, (64, 64, 64))   # 你的裁剪函数
        input_tensor = cropped_tensor.unsqueeze(0)
        return input_tensor

# 1. 严格用csv顺序生成影像路径
pd_img_paths = [os.path.join('PD', row[1]) for row in data_PD if os.path.exists(os.path.join('PD', row[1]))]
control_img_paths = [os.path.join('Control', row[1]) for row in data_control if os.path.exists(os.path.join('Control', row[1]))]
swedd_img_paths = [os.path.join('SWEDD', row[1]) for row in data_swedd if os.path.exists(os.path.join('SWEDD', row[1]))]
prodromal_img_paths = [os.path.join('Prodromal', row[1]) for row in data_prodromal if os.path.exists(os.path.join('Prodromal', row[1]))]

print(f'PD影像数量: {len(pd_img_paths)}')
print(f'Control影像数量: {len(control_img_paths)}')
print(f'SWEDD影像数量: {len(swedd_img_paths)}')
print(f'Prodromal影像数量: {len(prodromal_img_paths)}')

batch_size = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nii = CNN_3D(num_class=1)
nii = nii.to(device)

# 2. 用于影像特征提取
# PD组
dataset = NiiDataset(pd_img_paths)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
PD_output = torch.cat(all_outputs, dim=0)
print('PD nii shape--->', PD_output.shape)

# Control组
dataset = NiiDataset(control_img_paths)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
control_output = torch.cat(all_outputs, dim=0)
print('control nii shape--->', control_output.shape)

# Prodromal组
dataset = NiiDataset(prodromal_img_paths)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
prodromal_output = torch.cat(all_outputs, dim=0)
print('prodromal nii shape--->', prodromal_output.shape)

# SWEDD组
dataset = NiiDataset(swedd_img_paths)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
all_outputs = []
for batch_idx, batch_data in enumerate(dataloader):
    batch_data = batch_data.to(device)
    output = nii(batch_data)
    all_outputs.append(output)
swedd_output = torch.cat(all_outputs, dim=0)
print('swedd nii shape--->', swedd_output.shape)


PD影像数量: 125
Control影像数量: 132
SWEDD影像数量: 72
Prodromal影像数量: 80
PD nii shape---> torch.Size([125, 1])
control nii shape---> torch.Size([132, 1])
prodromal nii shape---> torch.Size([80, 1])
swedd nii shape---> torch.Size([72, 1])


In [34]:
# ===== 读取 DistMult 实体向量 =====
pd_df        = pd.read_csv('PD1.csv')
control_df   = pd.read_csv('control.csv')
prodromal_df = pd.read_csv('prodromal.csv')
swedd_df     = pd.read_csv('swedd.csv')

# 路径与文件名（根据你的脚本输出）
ENTITY_EMB_NPY = "entity_embeddings_distmult_PD.npy"
ENTITY2ID_TXT  = "entity2id_distmult_PD.txt"

# 1) 读实体→id 映射
entity2id = {}
with open(ENTITY2ID_TXT, "r", encoding="utf-8") as f:
    for line in f:
        ent, idx = line.strip().split("\t")
        entity2id[ent] = idx  # 注意 idx 是 str，后面会再映射到行号

# 2) 加载实体嵌入矩阵 (shape = [n_ent, 32])
entity_embeddings = torch.from_numpy(np.load(ENTITY_EMB_NPY)).float()

# 3) 把 id → 行号 做成 dict，方便索引
id_to_index = {id_str: idx for idx, id_str in enumerate(sorted(entity2id.values(), key=int))}

# 4) DistMultExtract：与之前 TransEextract 逻辑一致，只是名字改一下
class DistMultExtract:
    def __init__(self, entity_embeddings, entity2id, id_to_index):
        self.entity_embeddings = entity_embeddings
        self.entity2id = entity2id
        self.id_to_index = id_to_index

    def get_entity_embedding(self, entity):
        if entity in self.entity2id:
            ent_id = self.entity2id[entity]
            row_idx = self.id_to_index[ent_id]
            return self.entity_embeddings[row_idx]
        else:
            raise KeyError(f"Entity {entity} not found in entity2id mapping")

# 5) 初始化提取器
kg_extractor = DistMultExtract(entity_embeddings, entity2id, id_to_index)

# 6) 原先 get_embeddings() 函数保持不变，只把调用处的 `model=` 参数
#    换成新提取器 'kg_extractor'
def get_embeddings(df, extractor, nii_folder):
    embeddings_list = []
    skipped_rows = 0
    for _, row in df.iterrows():
        nii_file = row.iloc[1]
        nii_path = os.path.join(nii_folder, nii_file)
        if not os.path.exists(nii_path):
            skipped_rows += 1
            continue
        row_embeddings = []
        for col in df.columns[7:]:
            entity = str(row[col])
            if entity != '0' and entity in extractor.entity2id:
                row_embeddings.append(extractor.get_entity_embedding(entity))
        filename = str(row['Subject'])
        if filename in extractor.entity2id:
            row_embeddings.append(extractor.get_entity_embedding(filename))
        if row_embeddings:
            mean_embedding = torch.stack(row_embeddings).mean(dim=0)
            embeddings_list.append(mean_embedding)
    if not embeddings_list:
        return torch.empty((0, 32))
    return torch.stack(embeddings_list)

# 7) 获取四个 cohort 的 DistMult 嵌入
pd_transe        = get_embeddings(pd_df,        kg_extractor, 'PD')
control_transe   = get_embeddings(control_df,   kg_extractor, 'Control')
prodromal_transe = get_embeddings(prodromal_df, kg_extractor, 'Prodromal')
swedd_transe     = get_embeddings(swedd_df,     kg_extractor, 'SWEDD')

print(pd_transe.shape, control_transe.shape, prodromal_transe.shape, swedd_transe.shape)



torch.Size([125, 32]) torch.Size([132, 32]) torch.Size([80, 32]) torch.Size([72, 32])


In [35]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    
    for inputs, transe_embed, labels in loader:
        inputs, transe_embed, labels = inputs.to(device), transe_embed.to(device), labels.to(device)
        labels = labels.long()
        
        optimizer.zero_grad()
        outputs = model(inputs, transe_embed)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)

    # Convert lists to numpy arrays for easier manipulation
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    train_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    train_f1 = f1_score(all_labels, all_preds, average='macro')
    train_recall = recall_score(all_labels, all_preds, average='macro')
    train_precision = precision_score(all_labels, all_preds, average='macro')

    avg_loss = total_loss / len(loader)
    return avg_loss, train_auc, train_f1, train_recall, train_precision

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for inputs, transe_embed, labels in loader:
            inputs, transe_embed, labels = inputs.to(device), transe_embed.to(device), labels.to(device)
            labels = labels.long()
            
            outputs = model(inputs, transe_embed)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
    
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # 计算指标
    val_auc = roc_auc_score(all_labels, all_probs, multi_class='ovo')
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_recall = recall_score(all_labels, all_preds, average='macro')
    val_precision = precision_score(all_labels, all_preds, average='macro')
    
    # 计算平均损失
    avg_loss = total_loss / len(loader)
    return avg_loss, val_auc, val_f1, val_recall, val_precision


In [36]:
# ------- 重新拼接特征 + 构造 DataLoader ------- #
transe_embed_dim = 32   # DistMult 嵌入维度

X_pd = torch.cat([pd_EHR, PD_output.cpu(), pd_tensor, pd_transe], dim=1)
X_control = torch.cat([control_EHR, control_output.cpu(), control_tensor, control_transe], dim=1)
X_swedd = torch.cat([swedd_EHR, swedd_output.cpu(), swedd_tensor, swedd_transe], dim=1)
X_prodromal = torch.cat([prodromal_EHR, prodromal_output.cpu(), prodromal_tensor, prodromal_transe], dim=1)

y_pd = torch.zeros(len(X_pd))          # 标签顺序 PD=0
y_control = torch.ones(len(X_control)) # Control=1
y_swedd = torch.full((len(X_swedd),), 2)
y_prodromal = torch.full((len(X_prodromal),), 3)

X = torch.cat([X_pd[:120], X_control, X_prodromal, X_swedd], dim=0).float()
y = torch.cat([y_pd[:120], y_control, y_prodromal, y_swedd], dim=0).float()

features = X[:, :-transe_embed_dim]
kg_embed = X[:, -transe_embed_dim:]




In [37]:





from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, kg_train, kg_test = train_test_split(
    features.detach().cpu().numpy(),          # ←★ 改这里
    y.cpu().numpy(),
    kg_embed.detach().cpu().numpy(),          # ←★ 以及这里
    test_size=0.20, stratify=y.cpu().numpy(), random_state=32
)

X_train, X_val, y_train, y_val, kg_train, kg_val = train_test_split(
    X_train, y_train, kg_train,
    test_size=0.20, stratify=y_train, random_state=30
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
kg_train_tensor = torch.FloatTensor(kg_train).to(device)

X_val_tensor = torch.FloatTensor(X_val).to(device)
y_val_tensor = torch.LongTensor(y_val).to(device)
kg_val_tensor = torch.FloatTensor(kg_val).to(device)

X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)
kg_test_tensor = torch.FloatTensor(kg_test).to(device)

from torch.utils.data import TensorDataset, DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, kg_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   kg_val_tensor, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,  kg_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size)
print("DataLoaders ready:", len(train_loader), len(val_loader), len(test_loader))
# ------------------------------------------- #


DataLoaders ready: 9 3 3


In [1]:
# Cell A: New model for MRI + EHR + KG (replace 3-modality KGMultiModalTransformer)

class KGMultiModalTransformer2(nn.Module):
    """
    MRI + EHR + KG (DistMult/TransE embedding)
    inputs:  features [B, 2]  (ehr_scalar, mri_scalar)
             transe_embed [B, 32]
    output: logits [B, 4]
    """
    def __init__(self, embed_dim=32, num_heads=2, num_layers=2, transe_embed_dim=32):
        super().__init__()

        # two modality encoders
        self.ehr_encoder = nn.Sequential(
            nn.Linear(1, embed_dim),
            nn.ReLU()
        )
        self.img_encoder = nn.Sequential(
            nn.Linear(1, embed_dim),
            nn.ReLU()
        )

        # KG projection
        self.transe_proj = nn.Sequential(
            nn.Linear(transe_embed_dim, embed_dim),
            nn.ReLU()
        )

        # qkv for two modalities
        self.ehr_q = nn.Linear(embed_dim, embed_dim)
        self.ehr_k = nn.Linear(embed_dim, embed_dim)
        self.ehr_v = nn.Linear(embed_dim, embed_dim)

        self.img_q = nn.Linear(embed_dim, embed_dim)
        self.img_k = nn.Linear(embed_dim, embed_dim)
        self.img_v = nn.Linear(embed_dim, embed_dim)

        # cross-attention (same as your original)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        # fusion head (keep same output =4 for PPMI)
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 3, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 4)
        )

    def forward(self, inputs, transe_embed):
        """
        inputs: [B,2] -> ehr, mri
        transe_embed: [B,32]
        """
        ehr = inputs[:, 0:1]
        img = inputs[:, 1:2]

        ehr_feat = self.ehr_encoder(ehr).unsqueeze(1)  # [B,1,E]
        img_feat = self.img_encoder(img).unsqueeze(1)  # [B,1,E]

        transe_feat = self.transe_proj(transe_embed).unsqueeze(1)  # [B,1,E]

        # append KG token to each modality
        ehr_feat = torch.cat([ehr_feat, transe_feat], dim=1)  # [B,2,E]
        img_feat = torch.cat([img_feat, transe_feat], dim=1)

        # qkv
        Qe, Ke, Ve = self.ehr_q(ehr_feat), self.ehr_k(ehr_feat), self.ehr_v(ehr_feat)
        Qi, Ki, Vi = self.img_q(img_feat), self.img_k(img_feat), self.img_v(img_feat)

        # cross-attention both directions
        ehr_to_img, _ = self.cross_attention(query=Qe, key=Ki, value=Vi)  # [B,2,E]
        img_to_ehr, _ = self.cross_attention(query=Qi, key=Ke, value=Ve)

        # pool and fuse
        final_combined = torch.cat(
            [
                ehr_to_img.mean(dim=1),   # [B,E]
                img_to_ehr.mean(dim=1),   # [B,E]
                transe_feat.squeeze(1)    # [B,E]
            ],
            dim=1
        )  # [B,3E]

        return self.fusion(final_combined)



In [2]:
# Cell B: rebuild features for MRI + EHR + KG (drop Bio)

transe_embed_dim = 32   # DistMult embedding dim (same as before)

# 原来: torch.cat([pd_EHR, PD_output.cpu(), pd_tensor, pd_transe], dim=1)
# 现在: 去掉 pd_tensor / control_tensor / swedd_tensor / prodromal_tensor

X_pd        = torch.cat([pd_EHR, PD_output.cpu(), pd_transe], dim=1)
X_control   = torch.cat([control_EHR, control_output.cpu(), control_transe], dim=1)
X_swedd     = torch.cat([swedd_EHR, swedd_output.cpu(), swedd_transe], dim=1)
X_prodromal = torch.cat([prodromal_EHR, prodromal_output.cpu(), prodromal_transe], dim=1)

# labels (keep your order)
y_pd        = torch.zeros(len(X_pd))          
y_control   = torch.ones(len(X_control))      
y_swedd     = torch.full((len(X_swedd),), 2)
y_prodromal = torch.full((len(X_prodromal),), 3)

# 与你原来一样的拼接顺序
X = torch.cat([X_pd[:120], X_control, X_prodromal, X_swedd], dim=0).float()
y = torch.cat([y_pd[:120], y_control, y_prodromal, y_swedd], dim=0).float()

# split out KG
features = X[:, :-transe_embed_dim]   # 现在 features 只有 2 列：EHR + MRI
kg_embed = X[:, -transe_embed_dim:]   # 最后一段仍是 KG 32维

print("features shape:", features.shape)  # [N,2]
print("kg_embed shape:", kg_embed.shape)  # [N,32]
print("labels shape:", y.shape)


NameError: name 'pd_EHR' is not defined

In [ ]:
# Cell C: instantiate MRI + EHR + KG model

embed_dim        = 32
transe_embed_dim = 32
num_epochs       = 200
batch_size       = 32
learning_rate    = 1e-5
weight_decay     = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = KGMultiModalTransformer2(
    embed_dim=embed_dim,
    transe_embed_dim=transe_embed_dim,
    num_heads=2,
    num_layers=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print(model)


In [3]:
#Cell D
train_losses = []
train_aucs = []
train_f1s = []
train_recalls = []
train_precisions = []

val_losses = []
val_aucs = []
val_f1s = []
val_recalls = []
val_precisions = []

for epoch in range(num_epochs):
    train_loss, train_auc, train_f1, train_recall, train_precision = train_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_auc, val_f1, val_recall, val_precision = validate_epoch(
        model, val_loader, criterion, device
    )
    
    train_losses.append(train_loss)
    train_aucs.append(train_auc)
    train_f1s.append(train_f1)
    train_recalls.append(train_recall)
    train_precisions.append(train_precision)
    
    val_losses.append(val_loss)
    val_aucs.append(val_auc)
    val_f1s.append(val_f1)
    val_recalls.append(val_recall)
    val_precisions.append(val_precision)
    
    print(f"Epoch {epoch + 1}/{num_epochs}, "
          f"Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}  "
          f"Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")


NameError: name 'num_epochs' is not defined

In [4]:
# Cell E (REPLACE): Final ALL metrics on train/val/test + optional report/cm

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import label_binarize

def eval_loader(model, loader, device, num_classes=4):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for inputs, transe_embed, labels in loader:
            inputs = inputs.to(device)
            transe_embed = transe_embed.to(device)
            labels = labels.to(device).long()

            outputs = model(inputs, transe_embed)
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.detach().cpu().numpy())

    all_probs  = np.concatenate(all_probs, axis=0)    # [N,C]
    all_labels = np.concatenate(all_labels, axis=0)   # [N]
    preds = np.argmax(all_probs, axis=1)

    acc = accuracy_score(all_labels, preds)
    prec = precision_score(all_labels, preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, preds, average='macro', zero_division=0)

    # AUC-ROC (macro, ovr)
    try:
        y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
        auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
    except ValueError:
        auc = float("nan")

    return acc, prec, rec, f1, auc, all_labels, preds, all_probs


def print_metrics(tag, acc, prec, rec, f1, auc):
    print(f"\n=== {tag} Metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")


# ---------- Train ----------
train_acc, train_prec, train_rec, train_f1, train_auc, _, _, _ = eval_loader(
    model, train_loader, device, num_classes=4
)
print_metrics("Final Train", train_acc, train_prec, train_rec, train_f1, train_auc)

# ---------- Val ----------
val_acc, val_prec, val_rec, val_f1, val_auc, _, _, _ = eval_loader(
    model, val_loader, device, num_classes=4
)
print_metrics("Final Validation", val_acc, val_prec, val_rec, val_f1, val_auc)

# ---------- Test ----------
test_acc, test_prec, test_rec, test_f1, test_auc, test_labels, test_preds, test_probs = eval_loader(
    model, test_loader, device, num_classes=4
)
print_metrics("Final Test", test_acc, test_prec, test_rec, test_f1, test_auc)

# ---------- (Optional) Confusion Matrix + Report on Test ----------
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["PD", "Control", "SWEDD", "Prodromal"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("PPMI Confusion Matrix (Test)")
plt.show()

print("\nClassification Report (Test):")
print(classification_report(test_labels, test_preds, digits=4, zero_division=0))



NameError: name 'model' is not defined

In [ ]:
#Cell F
plt.figure(figsize=(10, 5))
plt.plot(range(len(train_aucs)), train_aucs, label="Train AUC", color="blue")
plt.plot(range(len(val_aucs)), val_aucs, label="Val AUC", color="red")
plt.title("PPMI KG+Transformer AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(len(train_losses)), train_losses, label="Train Loss", color="blue")
plt.plot(range(len(val_losses)), val_losses, label="Val Loss", color="red")
plt.title("PPMI KG+Transformer Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()
